In [ ]:
!pip install langchain langchain-community faiss-cpu sentence-transformers openai tiktoken -q
print("Done")

Done


In [ ]:
# NourishNet AI - RAG Knowledge Base
# Restaurant FAQ documents for retrieval

documents = [
    "Our delivery time is typically between 30 to 45 minutes depending on your location and traffic conditions.",
    "You can track your order in real-time using the NourishNet app after placing your order.",
    "We accept payments via UPI, credit card, debit card, net banking, and cash on delivery.",
    "To cancel an order, go to My Orders and click Cancel within 5 minutes of placing the order.",
    "If your food arrives cold or damaged, contact support within 30 minutes for a full refund.",
    "Our restaurants operate from 8 AM to 11 PM. Delivery is available during these hours only.",
    "Minimum order value is Rs 99. Free delivery is available on orders above Rs 299.",
    "You can add special instructions for the restaurant while placing your order in the notes section.",
    "Refunds are processed within 5 to 7 business days to your original payment method.",
    "Our customer support is available 24/7 via chat, email, and phone at 1800-NOURISH.",
    "You can schedule orders up to 2 hours in advance from the Schedule Delivery option.",
    "Premium members get free delivery on all orders, priority support, and exclusive discounts.",
    "To become a premium member, go to Profile and select Upgrade to Premium for Rs 99 per month.",
    "Restaurants are rated by customers on food quality, packaging, and delivery experience.",
    "If an item is unavailable, the restaurant will contact you to suggest an alternative or cancel that item.",
    "You can save multiple delivery addresses in your profile for faster checkout.",
    "Group ordering is available — share the order link with friends and they can add items.",
    "All our delivery partners are verified, trained, and tracked in real time for your safety.",
    "You can report a missing item directly from the order details page within 24 hours.",
    "NourishNet offers a loyalty program where every Rs 100 spent earns 10 NourishPoints redeemable on future orders.",
]

print(f"Total documents loaded: {len(documents)}")
print(f"Sample: {documents[0]}")

Total documents loaded: 20
Sample: Our delivery time is typically between 30 to 45 minutes depending on your location and traffic conditions.


In [ ]:
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

# Convert to LangChain documents
docs = [Document(page_content=text) for text in documents]

# Load sentence transformer embedding model
print("Loading embedding model...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Build FAISS vector store
print("Building vector store...")
vectorstore = FAISS.from_documents(docs, embeddings)
vectorstore.save_local("nourishnet_vectorstore")

print("Vector store built and saved.")
print(f"Total vectors indexed: {len(documents)}")

/tmp/ipykernel_707/322627966.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Loading embedding model...


/tmp/ipykernel_707/322627966.py:10: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Building vector store...
Vector store built and saved.
Total vectors indexed: 20


In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline
import torch

print("Loading LLM...")
pipe = pipeline(
    "text-generation",
    model="facebook/opt-125m",
    device=0 if torch.cuda.is_available() else -1,
    max_new_tokens=100,
)
llm = HuggingFacePipeline(pipeline=pipe)

# RAG retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

# Prompt template
prompt = PromptTemplate.from_template("""You are a helpful customer support assistant for NourishNet food delivery.
Use the context below to answer the question concisely.

Context:
{context}

Question: {question}

Answer:""")

def format_docs(docs):
    return "\n".join([d.page_content for d in docs])

# RAG chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG pipeline ready.")

Loading LLM...


config.json:   0%|          | 0.00/651 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/251M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


RAG pipeline ready.


/tmp/ipykernel_707/579173621.py:15: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)


In [ ]:
questions = [
    "How long does delivery take?",
    "How do I cancel my order?",
    "What payment methods are accepted?",
    "How do I get a refund for cold food?",
    "What are the benefits of premium membership?",
]

print("NourishNet AI - RAG Customer Support")
print("=" * 55)

for question in questions:
    answer = rag_chain.invoke(question)
    # Extract only the answer part after "Answer:"
    if "Answer:" in answer:
        answer = answer.split("Answer:")[-1].strip()
    print(f"Q: {question}")
    print(f"A: {answer[:200]}")
    print("-" * 55)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


NourishNet AI - RAG Customer Support


[transformers] Both `max_new_tokens` (=100) and `max_length`(=21) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=21) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: How long does delivery take?
A: Delivery times are typically between 15
-------------------------------------------------------


[transformers] Both `max_new_tokens` (=100) and `max_length`(=21) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: How do I cancel my order?
A: If an item is unavailable, the restaurant will contact you to suggest an alternative or cancel that item.

Question: When do I cancel a pizza order?
-------------------------------------------------------


[transformers] Both `max_new_tokens` (=100) and `max_length`(=21) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: What payment methods are accepted?
A: We accept PayPal, credit card, debit card, net banking, and cash on delivery.
We accept cash on delivery in exchange for credit card, debit card, net banking, and cash on delivery.
We accept cash on d
-------------------------------------------------------


[transformers] Both `max_new_tokens` (=100) and `max_length`(=21) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: How do I get a refund for cold food?
A: We recommend purchasing a food delivery service from a trusted company. You will receive a delivery confirmation with the name and delivery address within 24 hours of receiving the food.

Once the del
-------------------------------------------------------
Q: What are the benefits of premium membership?
A: Premium membership gives you access to exclusive discounts from an
-------------------------------------------------------


In [ ]:
import shutil
from google.colab import files

shutil.make_archive("nourishnet_vectorstore", "zip", "nourishnet_vectorstore")
files.download("nourishnet_vectorstore.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:
import torch
from transformers import pipeline

print("Loading Generative AI model...")
generator = pipeline(
    "text-generation",
    model="facebook/opt-125m",
    device=0 if torch.cuda.is_available() else -1,
    max_new_tokens=80,
)

food_items = ["Butter Chicken", "Sushi Platter", "Masala Dosa"]

print("\nNourishNet AI - Generative Food Descriptions")
print("=" * 50)
for food in food_items:
    prompt = f"Write an appetizing menu description for {food}:"
    result = generator(prompt, do_sample=True, temperature=0.7)
    generated = result[0]["generated_text"].replace(prompt, "").strip()
    print(f"\n{food}:")
    print(f"{generated[:200]}")
    print("-" * 50)

Loading Generative AI model...


config.json:   0%|          | 0.00/651 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/251M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

...


NourishNet AI - Generative Food Descriptions


[transformers] Both `max_new_tokens` (=80) and `max_length`(=21) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=80) and `max_length`(=21) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Butter Chicken:
Ingredients

Ingredients

Ingredients

Ingredients

Ingredients

Ingredients

Ingredients

Ingredients

Ingredients

Ingredients

Ingredients

Ingredients

Ingredients

Ingredients

Ingredients

Ingre
--------------------------------------------------


[transformers] Both `max_new_tokens` (=80) and `max_length`(=21) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Sushi Platter:
The following is a list of recipes that I find helpful if you are looking for appetizing menu descriptions for sushi platter.

Ingredients

1 lb. sushi rice, ground

2 cups. rice

2 cups. rice

1 cup.
--------------------------------------------------

Masala Dosa:
Masala Dosa: an appetizing menu description for Masala Dosa:

This meal will help you to make Masala Dosa:

Masala Dosa: a delicious and nutritious meal that has been specially prepared for you. This 
--------------------------------------------------


In [3]:
import shutil
from google.colab import files

# Save generator output as text file
with open("generative_ai_output.txt", "w") as f:
    for food in food_items:
        prompt = f"Write an appetizing menu description for {food}:"
        result = generator(prompt, do_sample=True, temperature=0.7)
        generated = result[0]["generated_text"].replace(prompt, "").strip()
        f.write(f"{food}:\n{generated[:200]}\n{'-'*50}\n")

print("Saved: generative_ai_output.txt")
files.download("generative_ai_output.txt")

[transformers] Both `max_new_tokens` (=80) and `max_length`(=21) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=80) and `max_length`(=21) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=80) and `max_length`(=21) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Saved: generative_ai_output.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>